# GFM/VSG + Cyber-Resilient FDI Detection

Runs the full simulation toolkit: GFM/VSG frequency response, a parameter ablation, ANN training for MPPT, RBF-SVM FDI detection, and an FDI feature ablation. Run the cells in order.

**Runtime:** ~1–2 minutes (the ANN training cell, using genuine Levenberg-Marquardt, is the slowest step).


## 1. Get the code

Run **one** of the two cells below, whichever applies to you.


In [ ]:
# Option A — you have this repository zipped on your computer:
# this will open a file picker; choose the zip file
from google.colab import files
uploaded = files.upload()

import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')
os.chdir('project')
print('Ready — working directory:', os.getcwd())


In [ ]:
# Option B — the code is on GitHub (uncomment and edit the URL, then run instead of Option A):
# !git clone <your-repo-url> project
# %cd project


## 2. Install dependencies

Colab already has numpy/scipy/scikit-learn, but this pins compatible versions.


In [ ]:
!pip install -q -r requirements.txt


## 3. Run everything

Prints the GFM/VSG frequency response, a parameter ablation, ANN training, FDI detection performance, and an FDI feature ablation — and saves every table to `results/results.xlsx` (one sheet each).


In [ ]:
!python3 run_all_experiments.py


## 3b. Generate plots

Writes the frequency-response, voltage-step-response, and confusion-matrix plots to `results/figures/`, and displays them below.


In [ ]:
!python3 make_plots.py

from IPython.display import Image, display
for f in ['frequency_response.png', 'voltage_step_response.png', 'confusion_matrix.png']:
    display(Image(filename=f'results/figures/{f}'))


## 3c. Download the results folder

Zips `results/` (the spreadsheet and the figures) and downloads it to your computer.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('results', 'zip', 'results')
files.download('results.zip')


## 4. (Optional) Explore individual pieces

Each module can be imported and inspected directly — useful if you want to plot a curve, change a parameter, or dig into one result.


In [ ]:
import sys, numpy as np
sys.path.insert(0, '.')

from src.vsg_control import frequency_response
import matplotlib.pyplot as plt

t, df_gfm, df_gfl, rocof_gfm, rocof_gfl = frequency_response(t_end=1.5)

plt.figure(figsize=(8,4))
plt.plot(t, df_gfm, label='GFM/VSG')
plt.plot(t, df_gfl, label='GFL (baseline)', linestyle='--')
plt.axhline(0, color='gray', linewidth=0.5)
plt.xlabel('Time (s)'); plt.ylabel('Δf (Hz)')
plt.title('Frequency response to a 50% active-power step')
plt.legend(); plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Example: FDI detector on a single feature vector
from src.fdi_detector import FDIDetector

det = FDIDetector()
res = det.train_and_evaluate(n_samples=1200, n_folds=10, seed=42)
print('F1-score:', round(res['f1_mean'], 3))

# f1 = peak voltage deviation, f2 = peak power deviation, f3 = max |dP/dt|
# try changing these values and see whether it's flagged as an attack (1) or normal (0):
print('Prediction for [60, 100, 15000]:', det.predict(60, 100, 15000))


## Notes on run-to-run variation

You may see the computed values change slightly between runs (e.g. 94.5% vs. 94.6% accuracy, or an AUC-ROC of 0.964 vs. 0.976). This is expected and comes from ordinary sources of variation — different NumPy/scikit-learn versions producing slightly different pseudo-random sequences from the same seed, and the fixed-step swing-equation integrator's sensitivity to the chosen time window. See `README.md` for details. None of this changes the qualitative conclusions.
